<a href="https://colab.research.google.com/github/Keroles-Sedhom/Kevo-11/blob/main/cheese_price_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import zipfile
import pandas as pd

# فك ضغط الملف
with zipfile.ZipFile('archive (3).zip', 'r') as zip_ref:
    zip_ref.extractall('extracted_data')

# البحث عن ملف الـ CSV وقراءته تلقائياً
import os
csv_file = [f for f in os.listdir('extracted_data') if f.endswith('.csv')][0]
df = pd.read_csv(os.path.join('extracted_data', csv_file))

# عرض أول 5 صفوف من البيانات
df.head()

,series_id,series_title,canonical_url,geography_type,geography_id,geography_label,observed_date,product_name,quantity_value,quantity_unit,quantity_name,price_amount,currency_code,normalized_price_amount,normalized_quantity_value,normalized_quantity_unit
0,american_cheese,American cheese,https://costinflation.com/indices/american-che...,postal_code,11385,"New York, NY - Ridgewood/Glendale",2026-07-13,"Amazon Grocery, Pasteurized Process American C...",16.0,ounces,16 ounces,2.48,USD,0.9300,0.375,pounds
1,american_cheese,American cheese,https://costinflation.com/indices/american-che...,postal_code,11385,"New York, NY - Ridgewood/Glendale",2026-07-13,"Amazon Saver, American Singles, Pasteurized Pr...",16.0,ounces,16 ounces,2.48,USD,0.9300,0.375,pounds
2,american_cheese,American cheese,https://costinflation.com/indices/american-che...,postal_code,11385,"New York, NY - Ridgewood/Glendale",2026-07-13,"American cheese slices , contains 9 ingredient...",32.0,ounces,32 ounces,42.99,USD,8.0606,0.375,pounds
3,american_cheese,American cheese,https://costinflation.com/indices/american-che...,postal_code,11385,"New York, NY - Ridgewood/Glendale",2026-07-13,BORDEN CHEESE SLICES AMERICAN SINGLES 12 OZ PA...,36.0,ounces,36 ounces,49.90,USD,8.3167,0.375,pounds
4,american_cheese,American cheese,https://costinflation.com/indices/american-che...,postal_code,11385,"New York, NY - Ridgewood/Glendale",2026-07-13,Borden Lactose Free American Singles 8 Oz (Pac...,24.0,ounces,24 ounces,49.90,USD,12.4750,0.375,pounds


In [4]:
print(df.columns)
print(df.shape)

Index(['series_id', 'series_title', 'canonical_url', 'geography_type',
       'geography_id', 'geography_label', 'observed_date', 'product_name',
       'quantity_value', 'quantity_unit', 'quantity_name', 'price_amount',
       'currency_code', 'normalized_price_amount', 'normalized_quantity_value',
       'normalized_quantity_unit'],
      dtype='object')
(12039, 16)


In [5]:
print(df['price_amount'].describe())

count    12039.000000
mean        40.729366
std         21.701374
min          1.180000
25%         29.990000
50%         42.990000
75%         49.990000
max         89.990000
Name: price_amount, dtype: float64


In [7]:
print(df[['observed_date', 'price_amount']].isnull().sum())

observed_date    0
price_amount     0
dtype: int64


In [8]:
# تحويل عمود التاريخ إلى نوع تاريخ (Datetime)
df['observed_date'] = pd.to_datetime(df['observed_date'])

# استخراج السنة والشهر واليوم كأعمدة جديدة عشان الموديل يفهمها
df['year'] = df['observed_date'].dt.year
df['month'] = df['observed_date'].dt.month
df['day'] = df['observed_date'].dt.day

# عرض أول صفوف بعد التعديل للتأكد
display(df[['observed_date', 'year', 'month', 'day', 'price_amount']].head())

,observed_date,year,month,day,price_amount
0,2026-07-13,2026,7,13,2.48
1,2026-07-13,2026,7,13,2.48
2,2026-07-13,2026,7,13,42.99
3,2026-07-13,2026,7,13,49.90
4,2026-07-13,2026,7,13,49.90


In [9]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# 1. تحديد المدخلات (Features) والهدف (Target)
X = df[['year', 'month', 'day']]
y = df['price_amount']

# 2. تقسيم الداتا إلى بيانات تدريب (Train) وبيانات اختبار (Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. إنشاء نموذج الـ Machine Learning وتدريبه
model = LinearRegression()
model.fit(X_train, y_train)

# 4. اختبار النموذج وتوقع الأسعار
predictions = model.predict(X_test)

print("Model trained successfully!")
print("First 5 predictions:", predictions[:5])

Model trained successfully!
First 5 predictions: [41.42951449 39.38102389 44.87152765 38.15192953 41.59394269]


In [10]:
score = model.score(X_test, y_test)
print(f"Model Accuracy (R2 Score): {score * 100:.2f}%")

Model Accuracy (R2 Score): 1.18%


In [11]:
from sklearn.preprocessing import LabelEncoder

# 1. تحويل نص اسم المنتج إلى أرقام
le = LabelEncoder()
df['product_name_encoded'] = le.fit_transform(df['product_name'])

# 2. إعادة تعريف المدخلات (Features) لتشمل نوع المنتج والكمية بالإضافة للتاريخ
X = df[['year', 'month', 'day', 'product_name_encoded', 'quantity_value']]
y = df['price_amount']

# 3. إعادة تدريب الموديل بنفس الطريقة القديمة
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = LinearRegression()
model.fit(X_train, y_train)

# 4. حساب الدقة الجديدة
new_score = model.score(X_test, y_test)
print(f"New Model Accuracy: {new_score * 100:.2f}%")

New Model Accuracy: 4.97%


In [13]:
from sklearn.ensemble import RandomForestRegressor

# إنشاء نموذج قوي يعتمد على الأشجار
rf_model = RandomForestRegressor(random_state=42)
rf_model.fit(X_train, y_train)

# حساب الدقة الجديدة
rf_score = rf_model.score(X_test, y_test)
print(f"Random Forest Accuracy: {rf_score * 100:.2f}%")

Random Forest Accuracy: 98.35%
